# 12 — Aethalometer

**Theme:** wavelength-resolved black carbon, and what the extra wavelengths buy
you.

An aethalometer measures light absorbed by particles collected on a filter. A
single wavelength gives a black-carbon mass concentration; measuring at several
wavelengths says something about what is *producing* the carbon, because fresh
combustion soot and wood smoke absorb differently across the spectrum.

In [ ]:
import aerosoltools as at

aeth = at.load_aethalometer_file("../../tests/data/Sample_Aethalometer.csv")

print("instrument:", aeth.instrument)
print("columns   :", list(aeth.data))

## Per-channel units

Each channel carries its own unit and dtype, because they are not all the same
quantity — the mass concentrations are in ng/m³, while the Ångström exponent is
dimensionless.

In [ ]:
aeth.column_units

## The wavelength channels

Five wavelengths, from ultraviolet to infrared. Each is a black-carbon
equivalent mass concentration derived from absorption at that wavelength.

In [ ]:
for name in ["uv_bcc", "blue_bcc", "green_bcc", "red_bcc", "ir_bcc"]:
    channel = getattr(aeth, name)
    print(f"{name:10s} mean {channel.mean():9.1f}  max {channel.max():9.1f}")

IR is the conventional reference for "black carbon" because soot dominates
absorption there. UV picks up additional absorption from organic material —
which is what makes the comparison between them informative.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
for name, colour in [("uv_bcc", "violet"), ("blue_bcc", "tab:blue"),
                     ("green_bcc", "tab:green"), ("red_bcc", "tab:red"),
                     ("ir_bcc", "black")]:
    ax.plot(aeth.time, getattr(aeth, name), label=name.replace("_bcc", "").upper(),
            color=colour, lw=1)
ax.set_ylabel("BC equivalent [ng/m3]")
ax.legend(ncol=5)
ax.set_title("Black carbon at five wavelengths")

## Source apportionment

A wide-format export splits the signal into a fossil-fuel and a biomass-burning
contribution, using the difference in absorption between UV and IR.

In [ ]:
print(f"fossil fuel : {aeth.fossil_bcc.mean():9.1f} ng/m3")
print(f"biomass     : {aeth.biomass_bcc.mean():9.1f} ng/m3")
print(f"IR total    : {aeth.ir_bcc.mean():9.1f} ng/m3")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(aeth.time, aeth.fossil_bcc, label="fossil fuel", color="dimgray")
ax.plot(aeth.time, aeth.biomass_bcc, label="biomass burning", color="darkorange")
ax.set_ylabel("BC equivalent [ng/m3]")
ax.legend()
ax.set_title("Apportioned black carbon")

## The Angstrom absorption exponent

The AAE summarises how steeply absorption falls with wavelength. Values near 1
indicate fresh fossil-fuel soot; higher values indicate wood smoke or other
organic-rich aerosol. It is the quantity the apportionment above is built on.

In [ ]:
print(f"AAE mean   : {aeth.aae.mean():.3f}")
print(f"AAE median : {aeth.aae.median():.3f}")
print(f"AAE range  : {aeth.aae.min():.3f} to {aeth.aae.max():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(aeth.time, aeth.aae, color="tab:purple", lw=1)
ax.axhline(1.0, ls="--", color="gray", label="~1: fossil-fuel soot")
ax.axhline(2.0, ls=":", color="saddlebrown", label="~2: biomass burning")
ax.set_ylabel("AAE [-]")
ax.legend()
ax.set_title("Angstrom absorption exponent over time")

## Channels that are not in the file

Narrow-format exports carry only the wavelength channels. Asking for something
absent names what is available instead of failing obscurely — worth knowing,
since which channels exist depends on how the instrument was configured.

In [ ]:
print("available channels:", [c for c in aeth.data.columns if c != "All data"])

## Everything else still applies

An `Aethalometer` is an ordinary time series, so activities, cropping and
summaries work as usual. Because the channels are separate columns rather than
a size distribution, summaries are per channel.

In [ ]:
aeth.mark_activities({
    "First half":  [(str(aeth.time[0]), str(aeth.time[len(aeth.time) // 2]))],
    "Second half": [(str(aeth.time[len(aeth.time) // 2 + 1]), str(aeth.time[-1]))],
})
aeth.summarize_activities()

---

**Next:** [13 — APS](13-aps.ipynb).